# Household Energy Monitoring & Automated Load Management using KNN

This notebook builds a two-stage K-Nearest Neighbours (KNN) system for a smart home. It reads electrical measurements, decides whether consumption is normal, and switches off non-essential appliances when consumption becomes unusually high or abnormal.

**System overview**

| Stage | Model | Question it answers | Output |
|---|---|---|---|
| 1 | Consumption model | How serious is the current consumption? | `Normal` / `Unusual_High` / `Abnormal` |
| 2 | Load-action model | What should the controller do? | `Keep_All_ON` / `Shed_Non_Essential` / `Critical_Shedding` |
| 3 | Control logic | Switch off appliances according to the predicted action | Appliance states after shedding |

**Notebook layout**

1. Generate a synthetic dataset
2. Train the consumption model
3. Test the consumption model
4. Train the load-action model
5. Test the load-action model
6. Combine both models and simulate automated load shedding

---

## 1. Data Generation

Labelled smart-meter data is hard to find, so this cell creates a **synthetic dataset of 500 readings** (seed `42` makes it reproducible) and saves it as `smart_home_power.csv`.

**Features**

| Feature | Description |
|---|---|
| `hour_of_day` | Hour of the reading (0-23) |
| `temperature_c` | Ambient temperature, normally distributed around 26.5 °C and limited to 18-38 °C |
| `fridge_on`, `lights_on` | **Essential loads**. The fridge is on 85% of the time; lights are mostly on in the evening and at night |
| `entertainment_on`, `ac_on`, `ev_charger_on`, `washing_machine_on`, `water_heater_on`, `microwave_on` | **Non-essential / deferrable loads**, each with a time-of-day or temperature-dependent chance of being on |
| `voltage_v` | Grid voltage, around 230 V (limited to 215-245 V) |
| `current_a` | Current draw, calculated as power / voltage |
| `total_power_watts` | Sum of every appliance's power draw (for example AC about 1800 W, EV charger about 3500 W, water heater about 2000 W) plus a base load of 50-150 W |

**Target labels (based on power only)**

| Total power | `consumption_status` | `load_action` |
|---|---|---|
| above 5000 W | `Abnormal` | `Critical_Shedding` |
| above 3500 W | `Unusual_High` | `Shed_Non_Essential` |
| 3500 W or below | `Normal` | `Keep_All_ON` |

In [9]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)
n_samples = 500

# 1. Feature Generation
hour_of_day = np.random.randint(0, 24, size=n_samples)
temperature_c = np.round(np.random.normal(loc=26.5, scale=4.5, size=n_samples).clip(18.0, 38.0), 1)

# Essential Loads (85% and time-based LED usage)
fridge_on = np.random.binomial(1, 0.85, n_samples)
lights_prob = np.where((hour_of_day >= 18) | (hour_of_day <= 6), 0.90, 0.30)
lights_on = np.random.binomial(1, lights_prob)

# Non-Essential / Deferrable Loads
entertainment_on = np.random.binomial(1, np.where((hour_of_day >= 12) & (hour_of_day <= 23), 0.50, 0.15))
ac_on = np.random.binomial(1, np.where(temperature_c > 28.0, 0.75, np.where(temperature_c > 24.0, 0.40, 0.10)))
ev_charger_on = np.random.binomial(1, np.where((hour_of_day >= 22) | (hour_of_day <= 6), 0.35, 0.05))
washing_machine_on = np.random.binomial(1, np.where((hour_of_day >= 8) & (hour_of_day <= 18), 0.25, 0.05))
water_heater_on = np.random.binomial(1, np.where(((hour_of_day >= 6) & (hour_of_day <= 9)) | ((hour_of_day >= 18) & (hour_of_day <= 21)), 0.40, 0.10))
microwave_on = np.random.binomial(1, np.where(np.isin(hour_of_day, [7, 8, 12, 13, 19, 20]), 0.30, 0.05))

# Power per appliance (Watts)
base_power = np.random.uniform(50, 150, n_samples)
p_fridge = fridge_on * np.random.normal(180, 25, n_samples)
p_lights = lights_on * np.random.normal(60, 10, n_samples)
p_ent = entertainment_on * np.random.normal(250, 30, n_samples)
p_ac = ac_on * np.random.normal(1800, 150, n_samples)
p_ev = ev_charger_on * np.random.normal(3500, 200, n_samples)
p_wm = washing_machine_on * np.random.normal(1200, 100, n_samples)
p_wh = water_heater_on * np.random.normal(2000, 150, n_samples)
p_mw = microwave_on * np.random.normal(1000, 80, n_samples)

total_power_watts = np.round(np.maximum(0, base_power + p_fridge + p_lights + p_ent + p_ac + p_ev + p_wm + p_wh + p_mw), 1)

# Standard grid voltage without artificial fault trips
voltage_v = np.round(np.random.normal(230.0, 3.0, n_samples).clip(215.0, 245.0), 1)
current_a = np.round(total_power_watts / voltage_v, 2)

# 2. POWER-ONLY THRESHOLD LOGIC
consumption_status = []
load_action = []

for p in total_power_watts:
    if p > 5000:
        consumption_status.append('Abnormal')
        load_action.append('Critical_Shedding')
    elif p > 3500:
        consumption_status.append('Unusual_High')
        load_action.append('Shed_Non_Essential')
    else:
        consumption_status.append('Normal')
        load_action.append('Keep_All_ON')

# 3. Create DataFrame and export
df = pd.DataFrame({
    'hour_of_day': hour_of_day,
    'temperature_c': temperature_c,
    'fridge_on': fridge_on,
    'lights_on': lights_on,
    'entertainment_on': entertainment_on,
    'ac_on': ac_on,
    'ev_charger_on': ev_charger_on,
    'washing_machine_on': washing_machine_on,
    'water_heater_on': water_heater_on,
    'microwave_on': microwave_on,
    'voltage_v': voltage_v,
    'current_a': current_a,
    'total_power_watts': total_power_watts,
    'consumption_status': consumption_status,
    'load_action': load_action
})

df.to_csv('smart_home_power.csv', index=False)
print("Dataset created successfully!")

Dataset created successfully!


## 2. Train the Consumption Model

This cell trains the first KNN classifier, which predicts the **consumption status**: `Normal`, `Unusual_High` or `Abnormal`.

**Steps**
1. Load `smart_home_power.csv`.
2. Use three power-related features as inputs: `total_power_watts`, `current_a` and `voltage_v`. The target is `consumption_status`.
3. Split the data **80% training / 20% testing**. `stratify=y` keeps the class proportions the same in both sets, which matters because `Normal` readings are much more common than the others.
4. **Scale the features** with `StandardScaler`. KNN works with distances, so a feature measured in thousands of watts would otherwise swamp one measured in amps. The scaler is fitted on the training data only and then applied to the test data.
5. Train `KNeighborsClassifier` with `n_neighbors=3`. Each reading is classified by a vote among its 3 closest training readings.
6. Evaluate with **accuracy** and a **confusion matrix**.
7. Save the model (`consump model`) and its scaler (`consump_scaler`) with `joblib`, so the same scaling can be applied to new data later.

**Result:** the accuracy is **0.95**. In the confusion matrix, rows are the true classes and columns the predictions, in the order `Abnormal`, `Normal`, `Unusual_High`. Only 5 of the 100 test readings are misclassified. A KNN is most likely to make mistakes for readings close to the 3500 W and 5000 W thresholds, where neighbours from two classes mix.

In [10]:
#importing libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,confusion_matrix

#reading the file
data=pd.read_csv('smart_home_power.csv')

#separating features into x and y
power_features = ['total_power_watts', 'current_a', 'voltage_v']
X = data[power_features]
y = data['consumption_status']
#splitting data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

#feature scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

#creating model
model=KNeighborsClassifier(n_neighbors=3)
model.fit(X_train_scaled,y_train)

#predicting
y_pred=model.predict(X_test_scaled)

#evaluations
accuracy=accuracy_score(y_test,y_pred)
matrix=confusion_matrix(y_test,y_pred)
print("Accuracy:",accuracy)
print("confusion matrix:\n",matrix)

#saving the model
saved_model=joblib.dump(model,"consump model")
saved_scaler=joblib.dump(scaler,"consump_scaler")

print("Model and Scaler saved successfully!")

Accuracy: 0.95
confusion matrix:
 [[10  0  1]
 [ 0 71  2]
 [ 0  2 14]]
Model and Scaler saved successfully!


## 3. Test the Consumption Model

This cell checks that the **saved consumption model** works on new data, as it would in real use.

**Steps**
1. Load the saved model (`consump model`) and scaler (`consump_scaler`).
2. Create 7 new readings that cover the whole range, from a quiet early morning (355 W) to a heavy morning load (5296 W).
3. Keep only the three power features and **scale them with the loaded scaler**. This is the same scaler used in training, so new data is scaled in exactly the same way.
4. Predict and print the status of each reading.

**Result:** the first two readings are `Normal`, the next two are `Unusual_High` (3850 W and 4750 W), and the last three are `Abnormal` (above 5000 W). This matches the power thresholds used to create the labels.

In [11]:
#importing neceessary libraries
import pandas as pd
import joblib

#loading model
loaded_model=joblib.load("consump model")
#loading the scaler
loaded_scaler=joblib.load("consump_scaler")

#formulating new data
new_data=pd.DataFrame([{'hour_of_day': 5,  'temperature_c': 24.4, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 0, 'ac_on': 0, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 230.6, 'current_a': 1.54,  'total_power_watts': 355.0},
    {'hour_of_day': 11, 'temperature_c': 29.0, 'fridge_on': 1, 'lights_on': 0, 'entertainment_on': 0, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 233.1, 'current_a': 8.05,  'total_power_watts': 1876.9},
    {'hour_of_day': 18, 'temperature_c': 28.5, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 1, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 1, 'microwave_on': 0, 'voltage_v': 228.9, 'current_a': 16.82, 'total_power_watts': 3850.5},
    {'hour_of_day': 22, 'temperature_c': 26.0, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 1, 'ac_on': 0, 'ev_charger_on': 1, 'washing_machine_on': 1, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 229.4, 'current_a': 20.71, 'total_power_watts': 4750.0},
    {'hour_of_day': 11, 'temperature_c': 26.8, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 0, 'ac_on': 0, 'ev_charger_on': 1, 'washing_machine_on': 1, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 228.3, 'current_a': 22.19, 'total_power_watts': 5066.0},
    {'hour_of_day': 6,  'temperature_c': 37.4, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 0, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 1, 'water_heater_on': 1, 'microwave_on': 0, 'voltage_v': 233.8, 'current_a': 22.33, 'total_power_watts': 5221.0},
    {'hour_of_day': 8,  'temperature_c': 27.9, 'fridge_on': 1, 'lights_on': 0, 'entertainment_on': 1, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 1, 'microwave_on': 1, 'voltage_v': 231.7, 'current_a': 22.86, 'total_power_watts': 5296.1}])

#features
power_features=['total_power_watts', 'current_a', 'voltage_v']
new_data_power=new_data[power_features]
#making predictions
new_scaled_data=loaded_scaler.transform(new_data_power)
predictions=loaded_model.predict(new_scaled_data)
print("predictions:",predictions)


predictions: ['Normal' 'Normal' 'Unusual_High' 'Unusual_High' 'Abnormal' 'Abnormal'
 'Abnormal']


## 4. Train the Load-Action Model

This cell trains the second KNN classifier, which decides **what the controller should do**: `Keep_All_ON`, `Shed_Non_Essential` or `Critical_Shedding`.

The process is the same as for the consumption model:
1. Load the dataset and use the same inputs: `total_power_watts`, `current_a` and `voltage_v`. The target is now `load_action`.
2. Split 80% / 20% with stratification.
3. Scale the features with a new `StandardScaler`, fitted on the training data only.
4. Train a `KNeighborsClassifier` with `n_neighbors=3`.
5. Evaluate with accuracy and a confusion matrix.
6. Save the model (`load_action model`) and its scaler (`load_action scaler`).

**Result:** the accuracy is **0.95**, with the same confusion matrix as the consumption model.

> **Note:** the results are identical because `load_action` and `consumption_status` are created from the same power thresholds and map one-to-one (`Normal` = `Keep_All_ON`, `Unusual_High` = `Shed_Non_Essential`, `Abnormal` = `Critical_Shedding`). The second model therefore learns the same pattern as the first. It would only become a separate decision-maker if the load action depended on other things, such as which appliances are on or the time of day.

In [12]:
#importing libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,confusion_matrix

#reading the file
data=pd.read_csv('smart_home_power.csv')

#separating features into x and y
power_features = ['total_power_watts', 'current_a', 'voltage_v']
X = data[power_features]
y = data['load_action']
#splitting data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

#feature scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

#creating model
model=KNeighborsClassifier(n_neighbors=3)
model.fit(X_train_scaled,y_train)

#predicting
y_pred=model.predict(X_test_scaled)

#evaluations
accuracy=accuracy_score(y_test,y_pred)
matrix=confusion_matrix(y_test,y_pred)
print("Accuracy:",accuracy)
print("confusion matrix:\n",matrix)

#saving the model
saved_model=joblib.dump(model,"load_action model")
saved_scaler=joblib.dump(scaler,"load_action scaler")

print("Model and Scaler saved successfully!")

Accuracy: 0.95
confusion matrix:
 [[10  0  1]
 [ 0 71  2]
 [ 0  2 14]]
Model and Scaler saved successfully!


## 5. Test the Load-Action Model

This cell checks that the **saved load-action model** works on new data.

**Steps**
1. Load the saved model (`load_action model`) and scaler (`load_action scaler`).
2. Use the same 7 test readings as in section 3, covering low, medium, high and very high consumption.
3. Scale the power features with the loaded scaler and predict.

**Result:** low-power readings give `Keep_All_ON`, the two mid-range readings (3850 W and 4750 W) give `Shed_Non_Essential`, and the three readings above 5000 W give `Critical_Shedding`.

These actions are what the control logic in the next section uses to decide which appliances to switch off.

In [13]:
#importing neceessary libraries
import pandas as pd
import joblib

#loading model
loaded_model=joblib.load("load_action model")
#loading the scaler
loaded_scaler=joblib.load("load_action scaler")

#formulating new data
new_data=pd.DataFrame([{'hour_of_day': 5,  'temperature_c': 24.4, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 0, 'ac_on': 0, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 230.6, 'current_a': 1.54,  'total_power_watts': 355.0},
    {'hour_of_day': 11, 'temperature_c': 29.0, 'fridge_on': 1, 'lights_on': 0, 'entertainment_on': 0, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 233.1, 'current_a': 8.05,  'total_power_watts': 1876.9},
    {'hour_of_day': 18, 'temperature_c': 28.5, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 1, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 1, 'microwave_on': 0, 'voltage_v': 228.9, 'current_a': 16.82, 'total_power_watts': 3850.5},
    {'hour_of_day': 22, 'temperature_c': 26.0, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 1, 'ac_on': 0, 'ev_charger_on': 1, 'washing_machine_on': 1, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 229.4, 'current_a': 20.71, 'total_power_watts': 4750.0},
    {'hour_of_day': 11, 'temperature_c': 26.8, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 0, 'ac_on': 0, 'ev_charger_on': 1, 'washing_machine_on': 1, 'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 228.3, 'current_a': 22.19, 'total_power_watts': 5066.0},
    {'hour_of_day': 6,  'temperature_c': 37.4, 'fridge_on': 1, 'lights_on': 1, 'entertainment_on': 0, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 1, 'water_heater_on': 1, 'microwave_on': 0, 'voltage_v': 233.8, 'current_a': 22.33, 'total_power_watts': 5221.0},
    {'hour_of_day': 8,  'temperature_c': 27.9, 'fridge_on': 1, 'lights_on': 0, 'entertainment_on': 1, 'ac_on': 1, 'ev_charger_on': 0, 'washing_machine_on': 0, 'water_heater_on': 1, 'microwave_on': 1, 'voltage_v': 231.7, 'current_a': 22.86, 'total_power_watts': 5296.1}])

#features
power_features=['total_power_watts', 'current_a', 'voltage_v']
new_data_power=new_data[power_features]
#making predictions
new_scaled_data=loaded_scaler.transform(new_data_power)
predictions=loaded_model.predict(new_scaled_data)
print("predictions:",predictions)


predictions: ['Keep_All_ON' 'Keep_All_ON' 'Shed_Non_Essential' 'Shed_Non_Essential'
 'Critical_Shedding' 'Critical_Shedding' 'Critical_Shedding']


## 6. Combined System: Consumption Monitoring + Automated Load Shedding

This final cell joins both models into one pipeline and simulates how the system would behave on live readings.

**Steps**
1. **Load both models and both scalers.** Each model has its own scaler.
2. **Create test readings:** a heavy night-time load with many appliances on (5586 W), a quiet early morning (276 W) and a moderate evening load (3121 W).
3. **Scale the power features separately for each model**, then run both models to get the consumption status and the load action.
4. **Apply the load-shedding logic** to each reading:
   - `Critical_Shedding` or `Shed_Non_Essential`: the AC and the EV charger are switched OFF.
   - `Keep_All_ON`: all appliances keep their current state.
   - The fridge and lights are essential and are never switched off.
5. **Print a report** with the power, the predicted status and action, the AC and EV charger states before and after shedding, and a log message.

**Result:** the 5586 W reading is classed as `Abnormal` and triggers critical shedding, so the AC and EV charger go from ON to OFF. The 276 W and 3121 W readings are `Normal`, so everything is kept ON.

> **Note:** the shedding logic only changes the AC and EV charger, even though the critical-shedding log message also mentions the water heater. The washing machine, entertainment system, water heater and microwave are not switched off by this code. Add them to the logic if you want them shed as well.

In [15]:
# Import necessary libraries
import pandas as pd
import joblib

# 1. Load both saved KNN models and their respective scalers
print("Loading models and scalers...")
consumption_model = joblib.load("consump model")
consumption_scaler = joblib.load("consump_scaler")

load_action_model = joblib.load("load_action model")
load_action_scaler = joblib.load("load_action scaler")

# 2. Define test data samples (including appliance states)
test_data = pd.DataFrame([
    {
        'hour_of_day': 22, 'temperature_c': 26.0, 'fridge_on': 1, 'lights_on': 1, 
        'entertainment_on': 1, 'ac_on': 1, 'ev_charger_on': 1, 'washing_machine_on': 1, 
        'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 229.4, 'current_a': 24.35,  
        'total_power_watts': 5586.0
    },
    {
        'hour_of_day': 5, 'temperature_c': 24.4, 'fridge_on': 1, 'lights_on': 1, 
        'entertainment_on': 0, 'ac_on': 0, 'ev_charger_on': 0, 'washing_machine_on': 0, 
        'water_heater_on': 0, 'microwave_on': 0, 'voltage_v': 230.6, 'current_a': 1.20,  
        'total_power_watts': 276.1
    },
    {
        'hour_of_day': 18, 'temperature_c': 28.5, 'fridge_on': 1, 'lights_on': 1, 
        'entertainment_on': 1, 'ac_on': 0, 'ev_charger_on': 0, 'washing_machine_on': 0, 
        'water_heater_on': 1, 'microwave_on': 0, 'voltage_v': 228.9, 'current_a': 13.64, 
        'total_power_watts': 3121.2
    }
])

# 3. Extract power features and scale them independently for each model
power_features = ['total_power_watts', 'current_a', 'voltage_v']
X_test_power = test_data[power_features]

X_scaled_consump = consumption_scaler.transform(X_test_power)
X_scaled_load = load_action_scaler.transform(X_test_power)

# 4. Make predictions
status_preds = consumption_model.predict(X_scaled_consump)
action_preds = load_action_model.predict(X_scaled_load)

# 5. Simulate Load Shedding Logic & Post-Shed States
ac_on_post = []
ev_on_post = []
control_logs = []

for action, ac, ev in zip(action_preds, test_data['ac_on'], test_data['ev_charger_on']):
    if action == 'Critical_Shedding':
        # Turn off all heavy non-essential loads
        ac_on_post.append(0)
        ev_on_post.append(0)
        control_logs.append("CRITICAL SHEDDING EXECUTED: All heavy non-essential loads (AC, EV, Water Heater) forcefully switched OFF.")
    elif action == 'Shed_Non_Essential':
        # Turn off deferrable loads like AC and EV charger
        ac_on_post.append(0)
        ev_on_post.append(0)
        control_logs.append("SHEDDING EXECUTED: Non-essential loads (AC, EV Charger) switched OFF.")
    else:
        # Keep original states
        ac_on_post.append(ac)
        ev_on_post.append(ev)
        control_logs.append("NOMINAL: All load operational states maintained. (Action: Keep All ON)")

# 6. Construct the final output DataFrame matching your desired format
output_df = pd.DataFrame({
    'total_power_watts': test_data['total_power_watts'],
    'Predicted_Anomaly_Status': status_preds,
    'Predicted_Load_Action': action_preds,
    'ac_on': test_data['ac_on'],
    'ac_on_post_shed': ac_on_post,
    'ev_charger_on': test_data['ev_charger_on'],
    'ev_charger_on_post_shed': ev_on_post,
    'Control_Action_Log': control_logs
})

# Set pandas display options to match notebook table widths
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.colheader_justify', 'center')

print("\n--- Automated Load Management Simulation Report ---")
print(output_df)

Loading models and scalers...

--- Automated Load Management Simulation Report ---
   total_power_watts Predicted_Anomaly_Status Predicted_Load_Action  ac_on  ac_on_post_shed  ev_charger_on  ev_charger_on_post_shed                                     Control_Action_Log                                    
0       5586.0               Abnormal            Critical_Shedding     1           0               1                   0             CRITICAL SHEDDING EXECUTED: All heavy non-essential loads (AC, EV, Water Heater) force...
1        276.1                 Normal                  Keep_All_ON     0           0               0                   0                                NOMINAL: All load operational states maintained. (Action: Keep All ON)
2       3121.2                 Normal                  Keep_All_ON     0           0               0                   0                                NOMINAL: All load operational states maintained. (Action: Keep All ON)
